# Quickstart: one DIII-D shot from coil currents and Thomson profiles

This notebook runs the published entry on one shot of the public DIII-D test fold and draws the predicted flux map. It needs a CPU, about 2 GB of RAM and about 5 minutes. The weights are fetched from Hugging Face at the pinned revision and every file is checked against its SHA-256. The challenge data are the organisers' (Sophelio, CC BY 4.0).

Run it locally from a clone of the repository, or in Colab (the first cell clones it).

In [ ]:
import os, subprocess, sys
if not os.path.exists('infer.py'):
    if not os.path.exists('fusion-equilibrium-challenge'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/anulum/fusion-equilibrium-challenge'], check=True)
    os.chdir('fusion-equilibrium-challenge')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-inference-cpu.txt', 'huggingface_hub', 'matplotlib'], check=True)
subprocess.run([sys.executable, 'fetch_weights.py'], check=True)

In [ ]:
from huggingface_hub import hf_hub_download
shot = 'd3d_shot_0047b7837b.parquet'
hf_hub_download('Sophelio/fusion-equilibrium-challenge', f'data/diii_d_public_test/{shot}', repo_type='dataset', local_dir='nb_data')

In [ ]:
env = dict(os.environ, OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', JAX_PLATFORMS='cpu', JAX_ENABLE_X64='1')
subprocess.run([sys.executable, '-B', 'infer.py', '--machine', 'd3d', '--data-root', 'nb_data/data', '--split', 'diii_d_public_test',
                '--workers', '1', '--limit', '1', '--out', 'nb_out/d3d_one_shot.npz'], check=True, env=env)

Compare the regenerated shot with the archive submitted to Codabench (#986068). On the hardware the entry was built on it is bit-identical; on other CPUs, Colab's included, the last bits of float32 may differ. The check fails only if the flux map moves by more than 1e-5 of its range or a scalar by more than 1e-4.

In [ ]:
subprocess.run([sys.executable, 'tools/ci_check_shot.py', 'nb_out/d3d_one_shot.npz'], check=True)

The archive holds, per shot, the flux map on the machine's 65 × 65 grid at every reconstruction time, and q95 and βN. Below: the predicted flux surfaces at three times of the discharge, and the two scalars over the shot.

In [ ]:
import numpy as np, pyarrow.parquet as pq, matplotlib.pyplot as plt
z = np.load('nb_out/d3d_one_shot.npz')
psi, q95, beta = z['shot_0000_psirz'], z['shot_0000_q95'], z['shot_0000_betaN']
row = pq.read_table(f'nb_data/data/diii_d_public_test/{shot}', columns=['efit_times', 'efit_grid_R', 'efit_grid_Z'])
t = np.asarray(row['efit_times'][0].as_py()); R = np.asarray(row['efit_grid_R'][0].as_py()); Z = np.asarray(row['efit_grid_Z'][0].as_py())
fig, axes = plt.subplots(1, 4, figsize=(15, 4.6), gridspec_kw={'width_ratios': [1, 1, 1, 1.6]})
for ax, i in zip(axes[:3], (len(t) // 5, len(t) // 2, 4 * len(t) // 5)):
    ax.contour(R, Z, psi[i], levels=24, cmap='plasma', linewidths=1)
    ax.set_aspect('equal'); ax.set_title(f't = {t[i]:.0f} ms'); ax.set_xlabel('R [m]')
axes[0].set_ylabel('Z [m]')
axes[3].plot(t, q95, label='q95'); axes[3].plot(t, beta, label='βN'); axes[3].set_xlabel('time [ms]'); axes[3].legend()
fig.suptitle('Predicted equilibrium, DIII-D public test shot 0 (no magnetic measurement used)'); plt.tight_layout()

To regenerate a whole fold, see the README: `infer.py` answers any directory of shot files from inputs alone. On the hardware the entry was built on, the result is bit-identical to the submitted archive; on other CPUs the last bits may differ (the repository's CI measures the difference).